# Binay Karki | Pokhara municipality-wide wet-season LST — MAP003

**Current working Colab, SAME permanent ID** `1Jf2JZwiIodAHMXn7Xr2XYvOifqnJ8FXv`.

This is a **new, unexecuted Earth Engine workflow**. It supersedes the active cells of the completed historical GEE V00–V08 notebook. The entire **original 97-cell V018 is preserved byte-identically** as a read-only PRE archive at [Drive](https://drive.google.com/file/d/1OIqegc2RLGNqgUuO1dyrl2WclrDJAkOe/view), and earlier Drive revisions remain available. No old scientific outputs are modified.

**Question this run will answer:** which fraction of the *entire historical Pokhara municipal polygon* intersects any 2026 Landsat 8/9 scene, and which fraction has **at least one actual, quality-screened wet-season land-surface temperature observation**? A scene footprint is not a valid temperature; gray pixels remain NoData.

**Two different coverage denominators:** percent of **municipal area** intersected by scene footprints, and percent of **municipal area** with cloud-screened, uncertainty-screened usable thermal pixels. Earlier V05 20.5719% was *existing study-grid valid pixel area / historical municipality area*, not an error in temperature units and not a citywide export. The 30 m raster **sampling grid** does not mean 30 m intrinsic thermal sensor resolution (Landsat TIRS is acquired at approximately 100 m, resampled to 30 m).

**User execution:** Run cells in order in your authenticated Google Colab. The first authentication cell may ask you to authorize Earth Engine and (later) Google Drive. A successful export task is **not** a completed raster until the QA cells read back its bytes. No new temperature is synthesized or interpolated.

- Source: Landsat Collection 2 Level 2 `L2SP` only (L8 + L9); preserve Binay's historical V05 QA thresholds, but the AOI changes to the municipal polygon.
- Study-aligned source window: 31 May – 12 August 2026 (start inclusive; `2026-08-13` exclusive), matching Binay's historical V05 field source acquisition window. Final thesis narrower 44-date/186-event R8 vs full 47-date/202-event source remains a separate scientific admission question.
- Municipal polygon: derived from **Rupesh's historical Pokhara Lekhnath NepalDD/Everest shapefile** (not certified 2026 administrative survey), pinned to SHA-256 `b7ed29c435f736d436150ce735692990b320bef1e64b9a77d046fd39736d19f5`.

References: [Google EE Python Colab](https://developers.google.com/earth-engine/guides/python_install-colab), [L8 C2 L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2), [L9 C2 L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2), [USGS LST](https://www.usgs.gov/landsat-missions/landsat-collection-2-surface-temperature), [GEE Image export](https://developers.google.com/earth-engine/apidocs/export-image-todrive).

## 1. Configuration and source identities

In [ ]:
from pathlib import Path
import os, io, json, csv, hashlib, time, sys, subprocess, datetime

STABLE_COLAB_DRIVE_ID = "1Jf2JZwiIodAHMXn7Xr2XYvOifqnJ8FXv"
PRE_ARCHIVE_DRIVE_ID = "1OIqegc2RLGNqgUuO1dyrl2WclrDJAkOe"
EE_PROJECT_ID = "just-a-project-5007"   # This is the project ID in Binay's archived executed notebook. Change only if access has changed.
START_DATE = "2026-05-31"
END_EXCLUSIVE = "2026-08-13"  # Includes August 12, matching the historic GEE-V05 source period.
GRID_CRS, GRID_SCALE_METRES = "EPSG:32644", 30
SOURCE_MUNICIPAL_DRIVE_ID = "1PxGcC1OMT8A7AZpa_a2HCUgKzURTyT7l"
SOURCE_MUNICIPAL_SHA256 = "b7ed29c435f736d436150ce735692990b320bef1e64b9a77d046fd39736d19f5"
SOURCE_MUNICIPAL_GIT_URL = "https://raw.githubusercontent.com/FabinGurung/jp-research-lab/main/researchers/binay-karki/gis/map002/pokhara_metropolitan_historical_RMP_NepalDD_to_WGS84.geojson"
ORIGINAL_3SITE_WET_RASTER_ID = "1djjbRqPqDbZDsIETgVi2zUBFRP5kDZuC"  # Read-only historical comparison; never overwritten.
OUTPUT_FOLDER_ID = "1PH3JhBcsSnDMxAx4B4-9y_50ZLEbm4fd"
OUTPUT_FOLDER_NAME = "BINAY_MAP003_CITYWIDE_WET_2026_EXPORTS"  # Earth Engine uses folder NAME, never ID.
RUN_KEY = "BINAY_MAP003_CITYWIDE_WET_LST_2026_V01"
NODATA = -9999.0
RUN_EXPORT = True  # Running the export cell starts one real Google Earth Engine batch task.
OUTPUT_DIR = Path('/content/binay_map003_citywide')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PALETTE = ['#2B073D','#446FE6','#22C9CD','#50ED67','#D8E83A','#FFA02B','#E04510','#860300']
EDGES_C = [5,10,15,20,25,30,35]
CLASS_NAMES = ['≤5','>5–10','>10–15','>15–20','>20–25','>25–30','>30–35','>35']
print('MAP003 configured; no source data or export is claimed ready yet.')

## 2. Install libraries in Colab; load exact source-controlled municipal polygon

In [ ]:
# This is a reproducible dependency-install cell, not a claim that Earth Engine is authenticated.
required = ['earthengine-api','google-api-python-client','rasterio','geopandas','pyproj','matplotlib','pandas','requests']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *required])
import requests, geopandas as gpd, ee, numpy as np, pandas as pd, rasterio
from rasterio.features import geometry_mask
from shapely.geometry import shape, mapping
from pyproj import Transformer
from IPython.display import display
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

raw = requests.get(SOURCE_MUNICIPAL_GIT_URL, timeout=60)
raw.raise_for_status()
raw_bytes = raw.content
actual_source_sha = hashlib.sha256(raw_bytes).hexdigest()
assert actual_source_sha == SOURCE_MUNICIPAL_SHA256, (
    f'Municipal polygon SHA mismatch: {actual_source_sha}. STOP: no GEE export.')
municipality_file = OUTPUT_DIR / 'pokhara_metropolitan_historical_RMP_NepalDD_to_WGS84.geojson'
municipality_file.write_bytes(raw_bytes)
municipality_geojson = json.loads(raw_bytes)
assert municipality_geojson['type']=='FeatureCollection' and len(municipality_geojson['features'])==1
municipality_geom = municipality_geojson['features'][0]['geometry']
assert municipality_geom['type'] in ('Polygon','MultiPolygon')
local_mun = gpd.read_file(municipality_file).to_crs(GRID_CRS)
assert local_mun.geometry.is_valid.all()
municipal_planar_km2 = float(local_mun.geometry.area.sum()/1e6)
assert 460.0 < municipal_planar_km2 < 470.0, f'Unexpected municipal polygon area: {municipal_planar_km2}'
print('Historic municipality source VERIFIED SHA:', actual_source_sha)
print('Geometry area in UTM 44N:', round(municipal_planar_km2,4), 'km²; not a current official boundary certificate')

## 3. Authenticate Earth Engine using Binay’s registered Cloud project

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT_ID)
except Exception as init_error:
    print('Earth Engine authentication required:', str(init_error)[:220])
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=EE_PROJECT_ID)
city = ee.Geometry(municipality_geom, proj='EPSG:4326', geodesic=False)
print('Earth Engine connected:', EE_PROJECT_ID)
print('AOI: Rupesh historical municipal polygon, not the old 3-site study buffer.')

## 4. Discover and record every actual Landsat 8/9 source scene

In [ ]:
def collection(dataset_id):
    return (ee.ImageCollection(dataset_id)
      .filterBounds(city)
      .filterDate(START_DATE, END_EXCLUSIVE)
      .filter(ee.Filter.eq('PROCESSING_LEVEL','L2SP')))

l8 = collection('LANDSAT/LC08/C02/T1_L2')
l9 = collection('LANDSAT/LC09/C02/T1_L2')
raw_scenes = l8.merge(l9).sort('system:time_start')
scene_count = int(raw_scenes.size().getInfo())
if scene_count == 0:
    raise RuntimeError('No L2SP scenes intersect municipal AOI in this period: STOP (no synthetic temperatures).')

props = ['LANDSAT_PRODUCT_ID','SPACECRAFT_ID','system:time_start','WRS_PATH','WRS_ROW','CLOUD_COVER']
arrays = {k:raw_scenes.aggregate_array(k).getInfo() for k in props}
scene_records=[]
for idx in range(scene_count):
    r={k:arrays[k][idx] if idx<len(arrays[k]) else None for k in props}
    ts=r.get('system:time_start')
    r['acquired_UTC']=datetime.datetime.fromtimestamp(ts/1000, datetime.timezone.utc).isoformat() if ts else None
    scene_records.append(r)
scene_df=pd.DataFrame(scene_records).sort_values(['acquired_UTC','LANDSAT_PRODUCT_ID'])
scene_csv=OUTPUT_DIR / f'{RUN_KEY}_EXACT_LANDSAT_SCENES.csv'
scene_df.to_csv(scene_csv, index=False)
print('Real 2026 historical municipal AOI scene counts: L8',l8.size().getInfo(), 'L9',l9.size().getInfo(),'combined',scene_count)
display(scene_df.head(15))
print('Full scene manifest CSV:',scene_csv)

## 5. Preserve GEE-V05 surface temperature scaling and quality screening

In [ ]:
# The archived V05 already applied these thresholds: reject QA_PIXEL bits 0–5;
# QA_RADSAT == 0; ST_QA <= 5 Kelvin; ST_CDIST >= 1 km; LST 15–60°C.
# Reusing QA gates avoids silently changing Binay's 2026 satellite scientific criteria.
V05_MAX_ST_QA_K = 5.0
V05_MIN_ST_CDIST_KM = 1.0
V05_LST_MIN_C, V05_LST_MAX_C = 15.0, 60.0

def prep_citywide(image):
    qa = image.select('QA_PIXEL')
    bad = qa.bitwiseAnd(63).neq(0)   # bits 0,1,2,3,4,5 = fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    not_saturated = image.select('QA_RADSAT').eq(0)
    lst_c = image.select('ST_B10').multiply(0.00341802).add(149).subtract(273.15)
    st_qa_k = image.select('ST_QA').multiply(0.01)
    cloud_distance_km = image.select('ST_CDIST').multiply(0.01)
    base_mask = bad.Not().And(not_saturated)
    primary_mask = (base_mask.And(st_qa_k.lte(V05_MAX_ST_QA_K))
                    .And(cloud_distance_km.gte(V05_MIN_ST_CDIST_KM))
                    .And(lst_c.gte(V05_LST_MIN_C)).And(lst_c.lte(V05_LST_MAX_C)))
    return ee.Image.cat([
        lst_c.updateMask(base_mask).rename('LST_BASE_C'),
        lst_c.updateMask(primary_mask).rename('LST_PRIMARY_C'),
    ]).copyProperties(image,['system:time_start','LANDSAT_PRODUCT_ID','SPACECRAFT_ID'])

processed = raw_scenes.map(prep_citywide)
city_median_lst = processed.select('LST_PRIMARY_C').median().rename('LST_C').clip(city)
n_primary = processed.select('LST_PRIMARY_C').count().rename('N_PRIMARY').unmask(0).clip(city)
n_base = processed.select('LST_BASE_C').count().rename('N_BASE').unmask(0).clip(city)
# A scene's geographic footprint is not proof of usable thermal measurements.
footprints = raw_scenes.map(lambda im: ee.Image.constant(1).rename('FP').clip(im.geometry()))
scene_footprint = footprints.max().unmask(0).rename('SCENE_FOOTPRINT').clip(city)
print('Prepared server-side citywide median with real masks and per-pixel observations; NOT yet exported.')

## 6. Three separate municipal coverage metrics — report square kilometres and percent of area

In [ ]:
# Calculate area using the same UTM grid/scale for every numerator and denominator.
pxarea = ee.Image.pixelArea().clip(city)
area_bands = ee.Image.cat([
    pxarea.rename('AREA_CITY_M2'),
    pxarea.updateMask(scene_footprint.gt(0)).rename('AREA_SCENE_FOOTPRINT_M2'),
    pxarea.updateMask(n_base.gt(0)).rename('AREA_BASE_QA_LST_M2'),
    pxarea.updateMask(n_primary.gt(0)).rename('AREA_PRIMARY_QA_LST_M2'),
])
area_dict = area_bands.reduceRegion(reducer=ee.Reducer.sum(), geometry=city,
                                   scale=GRID_SCALE_METRES, crs=GRID_CRS,
                                   maxPixels=1000000000, tileScale=4).getInfo()
city_area_m2 = float(area_dict['AREA_CITY_M2'])
assert city_area_m2>0

def fraction(key):
    area=float(area_dict.get(key) or 0)
    pct=100.0*area/city_area_m2
    assert 0<=pct<=100.5, (key,pct)
    return {'area_km2':round(area/1e6,4),'municipal_area_pct':round(pct,3)}

coverage_summary={
    'status':'SERVER_EE_CALCULATED_NOT_YET_RASTER_VERIFIED',
    'source_municipal_sha256': SOURCE_MUNICIPAL_SHA256,
    'municipal_grid_area_km2':round(city_area_m2/1e6,4),
    'municipal_planar_reference_km2':round(municipal_planar_km2,4),
    'scene_footprint':fraction('AREA_SCENE_FOOTPRINT_M2'),
    'base_qa_valid_lst':fraction('AREA_BASE_QA_LST_M2'),
    'strict_v05_primary_valid_lst':fraction('AREA_PRIMARY_QA_LST_M2'),
    'number_of_scenes':scene_count,
    'start_inclusive':START_DATE, 'end_exclusive':END_EXCLUSIVE,
    'percentage_denominator':'100 × covered 30m projected pixel area / all municipal projected pixel area',
    'temperature_unit':'deg C', 'area_unit':'km2',
    'intrinsic_thermal_resolution_note':'Landsat 8/9 TIRS nominal 100m; product grid resampled to 30m',
    'method':'Binay V05 primary QA reused; changed AOI from 3-site buffer to full historical municipality',
}
print(json.dumps(coverage_summary,indent=2))
print('Old V05 20.57% was original study-grid valid-cell area / city, NOT the result of this new export.')
coverage_json=OUTPUT_DIR/f'{RUN_KEY}_AREA_COVERAGE_PRE_EXPORT.json'
coverage_json.write_text(json.dumps(coverage_summary,indent=2),encoding='utf-8')

## 7. Start the real full-city GeoTIFF export — **requires authenticated Earth Engine**

In [ ]:
# Distinct GeoTIFF bands: observed LST_C, strict valid observation count,
# base QA observation count, and scene footprint. -9999 is NoData, never 0°C.
# The export covers the municipal BOUNDING BOX; all outside-municipal cells remain NoData.
export_image=(ee.Image.cat([city_median_lst,n_primary,n_base,scene_footprint])
              .clip(city).unmask(NODATA,sameFootprint=False).toFloat())

if RUN_EXPORT:
    active_states={'READY','RUNNING','SUBMITTED','UNSUBMITTED'}
    duplicates=[t for t in ee.batch.Task.list()
                if t.status().get('description') == RUN_KEY
                and t.status().get('state') in active_states]
    if duplicates:
        raise RuntimeError(f'Existing active MAP003 export task detected: {duplicates[0].id}; monitor it instead of starting duplicate.')
    task=ee.batch.Export.image.toDrive(
        image=export_image, description=RUN_KEY, folder=OUTPUT_FOLDER_NAME,
        fileNamePrefix=RUN_KEY, region=city.bounds(maxError=1),
        scale=GRID_SCALE_METRES,crs=GRID_CRS, maxPixels=10000000000,
        fileFormat='GeoTIFF', formatOptions={'cloudOptimized':True, 'noData':NODATA})
    task.start()
    print('EARTH ENGINE TASK STARTED:',task.id,'STATE:',task.status().get('state'))
    print('This is NOT a completed file. Continue after Earth Engine reports COMPLETED.')
else:
    task=None
    print('RUN_EXPORT=False — no Earth Engine batch task started.')

## 8. Inspect real task status; never infer completion from a submitted request

In [ ]:
if task is None:
    print('No new task in this session. Switch RUN_EXPORT=True or inspect an earlier GEE task.')
else:
    current_status=task.status()
    display(pd.DataFrame([current_status]))
    if current_status.get('state') != 'COMPLETED':
        print('STOP before downloading: task state =',current_status.get('state'))
    else:
        print('Earth Engine says COMPLETED. Continue to Google Drive byte readback and pixel QA.')

## 9. Download the **actual completed GeoTIFF** from its registered Google Drive folder

In [ ]:
# This cell uses Google Drive authorization only for retrieving the output;
# if Colab's browser credential-propagation fails, retry authenticated Colab or
# download the named GeoTIFF manually into OUTPUT_DIR before running section 10.
from google.colab import auth as colab_auth
from google.auth import default

try:
    colab_auth.authenticate_user()
    credentials,_=default(scopes=['https://www.googleapis.com/auth/drive'])
    drive_api=build('drive','v3',credentials=credentials,cache_discovery=False)
    folder_meta=drive_api.files().get(fileId=OUTPUT_FOLDER_ID, fields='id,name,mimeType,parents',supportsAllDrives=True).execute()
    assert folder_meta['name'] == OUTPUT_FOLDER_NAME and folder_meta['mimeType']=='application/vnd.google-apps.folder'
    print('Verified export folder stable ID:',OUTPUT_FOLDER_ID)
except Exception as e:
    raise RuntimeError('Drive OAuth unavailable in this Colab runtime. Reauthorize or manually download TIFF before Section 10.') from e

if task is not None and task.status().get('state')!='COMPLETED':
    raise RuntimeError('Export not completed: do not treat task submission as a raster.')

files=drive_api.files().list(q=f"'{OUTPUT_FOLDER_ID}' in parents and name = '{RUN_KEY}.tif' and trashed = false",
                fields='files(id,name,size,md5Checksum,mimeType,parents),nextPageToken',pageSize=50).execute()['files']
if len(files)!=1:
    raise RuntimeError(f'Expected EXACTLY ONE {RUN_KEY}.tif in registered folder; found {len(files)}. Inspect export status/folder duplicates.')
file_meta=files[0]
assert int(file_meta['size'])>10000
raster_path=OUTPUT_DIR/file_meta['name']
req=drive_api.files().get_media(fileId=file_meta['id'])
with raster_path.open('wb') as fp:
    downloader=MediaIoBaseDownload(fp,req)
    complete=False
    while not complete:
        _,complete=downloader.next_chunk()
sha256_raster=hashlib.sha256(raster_path.read_bytes()).hexdigest()
print('Completed provider GeoTIFF:',file_meta['id'],raster_path.stat().st_size,'bytes; SHA256:',sha256_raster)

## 10. Independent **raster-byte** QA and exact class-area ledger

In [ ]:
from rasterio.features import geometry_mask
from shapely.geometry import mapping
with rasterio.open(raster_path) as ds:
    assert ds.crs.to_string()==GRID_CRS and ds.count==4, (ds.crs,ds.count)
    assert abs(ds.res[0]-30)<0.1 and abs(ds.res[1]-30)<0.1, ds.res
    a=ds.read(1); n=ds.read(2); base=ds.read(3); fp=ds.read(4)
    muni_geom_raster=local_mun.to_crs(ds.crs).geometry.iloc[0]
    inside=geometry_mask([mapping(muni_geom_raster)],out_shape=(ds.height,ds.width),transform=ds.transform,invert=True)
    valid=inside & np.isfinite(a) & (a != NODATA) & (n>0)
    footprint=inside & (fp>0) & (fp!=NODATA)
    base_valid=inside & (base>0) & (base!=NODATA)
    pixel_km2=ds.res[0]*ds.res[1]/1e6
    municipal_pixels=int(inside.sum())
    valid_count=int(valid.sum())
    footprint_count=int(footprint.sum())
    bins=np.digitize(a[valid],EDGES_C,right=True)
    bin_counts=np.bincount(bins,minlength=8)
    assert len(bin_counts)==8 and bin_counts.sum()==valid_count
    assert valid_count<=footprint_count+2000, 'More usable LST pixels than scene footprint; investigate geometric edge effects'
    raster_qa={
      'raster_sha256':sha256_raster,'original_drive_id':file_meta['id'],
      'rows':ds.height,'columns':ds.width,'crs':ds.crs.to_string(),'resolution_m':[float(v) for v in ds.res],
      'municipal_pixels_center_rule':municipal_pixels,
      'scene_footprint_pixel_count':footprint_count,
      'valid_strict_lst_pixel_count':valid_count,
      'valid_strict_area_km2_30m_grid_est':round(valid_count*pixel_km2,4),
      'municipal_area_km2_30m_grid_est':round(municipal_pixels*pixel_km2,4),
      'valid_strict_pct_pixel_center_coverage':round(100*valid_count/municipal_pixels,4),
      'source_EE_area_pct_server':coverage_summary['strict_v05_primary_valid_lst']['municipal_area_pct'],
      'scene_footprint_pct_pixel_center':round(100*footprint_count/municipal_pixels,4),
      'class_counts':bin_counts.astype(int).tolist(),'class_hex':PALETTE,
      'all_noncovered_area_is_nodata':True,
      'thermal_sampling_grid_m':30,
      'native_sensor_nominal_thermal_resolution_m':100
    }
    # EE region area weights edges differently than pixel-centre membership; small discrepancies are expected.
    difference=abs(raster_qa['valid_strict_pct_pixel_center_coverage']-float(raster_qa['source_EE_area_pct_server']))
    assert difference < 1.0, f'EE area vs local masked pixel-area discrepancy >1 percentage point: {difference:.3f}'

qa_path=OUTPUT_DIR/f'{RUN_KEY}_RASTER_QA.json'
qa_path.write_text(json.dumps(raster_qa,indent=2))
class_df=pd.DataFrame({'class_index':range(1,9),'temp_bin_C':CLASS_NAMES,'palette_hex':PALETTE,
                       'valid_pixel_count':bin_counts,'area_km2_30m_grid_est':np.round(bin_counts*pixel_km2,4)})
class_csv=OUTPUT_DIR/f'{RUN_KEY}_EXACT_8_CLASS_COUNTS.csv'
class_df.to_csv(class_csv,index=False)
display(class_df)
print('Independent citywide pixel QA:',json.dumps(raster_qa,indent=2))

## 11. Reproduce the map with Binay’s exact eight-class palette; missing areas remain gray

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from rasterio.plot import plotting_extent

with rasterio.open(raster_path) as ds:
    arr=ds.read(1)
    muni=local_mun.to_crs(ds.crs)
    inside=geometry_mask([mapping(muni.geometry.iloc[0])],out_shape=(ds.height,ds.width),transform=ds.transform,invert=True)
    valid=inside & (arr!=NODATA) & np.isfinite(arr) & (ds.read(2)>0)
    color_index=np.digitize(arr,EDGES_C,right=True)
    canvas=np.ma.array(color_index,mask=~valid)
    cm=ListedColormap(PALETTE)
    cm.set_bad('#E9ECEF')
    extent=plotting_extent(ds)
fig,ax=plt.subplots(figsize=(16.53,11.69))
fig.patch.set_facecolor('white');ax.set_facecolor('white')
ax.imshow(canvas,extent=extent,origin='upper',cmap=cm,vmin=-0.5,vmax=7.5,interpolation='nearest')
mun.boundary.plot(ax=ax,color='#1E3648',linewidth=.8)
ax.set_aspect('equal');ax.set_xticks([]);ax.set_yticks([])
ax.set_title('Pokhara | wet-season Landsat land-surface temperature (2026)',fontsize=20,loc='left',pad=15)
subtitle=(f"Historical municipal footprint · {raster_qa['valid_strict_pct_pixel_center_coverage']:.2f}% municipality valid V05-QA LST"
          f" · {raster_qa['scene_footprint_pct_pixel_center']:.2f}% covered by acquired scenes")
fig.text(.10,.89,subtitle,fontsize=10.4)
fig.text(.10,.855,'30 m delivered raster sampling; Landsat 8/9 thermal observations acquired at ~100 m (not 30 m native).',fontsize=9.1,color='#425466')
fig.text(.10,.825,'Gray = no usable LST (not a cold-temperature bin); median of masked valid source observations, never interpolation.',fontsize=9.1,color='#9D3828')
handles=[Patch(facecolor=c,edgecolor='#657384',label=f'{label} °C') for c,label in zip(PALETTE,CLASS_NAMES)]
ax.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,-.12),ncol=8,frameon=False,fontsize=8)
fig.text(.10,.047,'Source: Binay V05 Landsat QA + RMP historical municipality polygon. Scientific validation remains separate.',fontsize=8.8,color='#55616F')
fig.tight_layout(rect=[.05,.10,.95,.83])
pdf_path=OUTPUT_DIR/f'{RUN_KEY}_TRUE_CITYWIDE_A3.pdf'
png_path=OUTPUT_DIR/f'{RUN_KEY}_TRUE_CITYWIDE_A3.png'
fig.savefig(pdf_path,facecolor='white',bbox_inches='tight')
fig.savefig(png_path,facecolor='white',dpi=300,bbox_inches='tight')
plt.show();plt.close(fig)
print('Source-derived map:',pdf_path,'and',png_path,'— NO AI-generated temperature pixels')

## 12. Export exact citywide map-data rows for Google Sheets / CSV audit

In [ ]:
# Only valid original LST values are written; no rows for masked or interpolated pixels.
# This may produce hundreds of thousands of rows; one native Google Sheet is supported
# only if the resulting cell count remains under Google's applicable sheet limits.
with rasterio.open(raster_path) as ds:
    a=ds.read(1)
    n=ds.read(2)
    mun=local_mun.to_crs(ds.crs)
    inside=geometry_mask([mapping(mun.geometry.iloc[0])],out_shape=(ds.height,ds.width),transform=ds.transform,invert=True)
    valid=inside & (a!=NODATA) & np.isfinite(a) & (n>0)
    rows,cols=np.where(valid)
    xs,ys=rasterio.transform.xy(ds.transform,rows,cols,offset='center')
    longitudes,latitudes=Transformer.from_crs(ds.crs,'EPSG:4326',always_xy=True).transform(xs,ys)
    vals=a[rows,cols]; num=n[rows,cols]; cl=np.digitize(vals,EDGES_C,right=True)+1
ledger_csv=OUTPUT_DIR/f'{RUN_KEY}_ORIGINAL_VALID_PIXELS.csv'
with ledger_csv.open('w',newline='',encoding='utf-8') as stream:
    writer=csv.writer(stream)
    writer.writerow(['pixel_row','pixel_col','longitude_wgs84','latitude_wgs84','LST_C','n_valid_2026_scenes','eight_class_index','source_run'])
    for r,c,lon,lat,lst,nscene,cindex in zip(rows,cols,longitudes,latitudes,vals,num,cl):
        writer.writerow([int(r),int(c),f'{lon:.8f}',f'{lat:.8f}',f'{lst:.6f}',int(nscene),int(cindex),RUN_KEY])
print('Exact valid pixel rows:',len(rows),'CSV bytes:',ledger_csv.stat().st_size,'SHA256:',hashlib.sha256(ledger_csv.read_bytes()).hexdigest())
assert len(rows)==raster_qa['valid_strict_lst_pixel_count']

## 13. Preserve each map, QA file and raw CSV in the dedicated Drive output folder

In [ ]:
# Fail closed on an existing filename with DIFFERENT bytes. We do not mutate
# the archived study-grid V05 outputs or previously completed thesis assets.
def drive_file_sha256(file_id):
    req=drive_api.files().get_media(fileId=file_id)
    memory=io.BytesIO(); dl=MediaIoBaseDownload(memory,req); done=False
    while not done: _,done=dl.next_chunk()
    return hashlib.sha256(memory.getvalue()).hexdigest()

def upload_or_verify_unchanged(path):
    digest=hashlib.sha256(path.read_bytes()).hexdigest()
    safe=path.name.replace("'","\'")
    present=drive_api.files().list(q=f"'{OUTPUT_FOLDER_ID}' in parents and name = '{safe}' and trashed = false",
                fields='files(id,name,size,md5Checksum,parents)',pageSize=100).execute()['files']
    if len(present)>1:raise RuntimeError('Ambiguous multiple output files: '+path.name)
    if present:
        result=present[0]
        if int(result['size'])!=path.stat().st_size or drive_file_sha256(result['id'])!=digest:
            raise RuntimeError('Filename collision with different bytes: '+path.name)
        state='REUSED_IDENTICAL'
    else:
        created=drive_api.files().create(body={'name':path.name,'parents':[OUTPUT_FOLDER_ID]},
          media_body=MediaFileUpload(str(path),resumable=True),fields='id,name,size,parents').execute()
        result=created
        assert drive_file_sha256(result['id'])==digest
        state='UPLOADED_VERIFIED'
    return {'file_name':path.name,'drive_file_id':result['id'],'size_bytes':int(result['size']),
            'sha256':digest,'status':state}

paths=[scene_csv,coverage_json,qa_path,class_csv,pdf_path,png_path,ledger_csv]
receipts=[upload_or_verify_unchanged(Path(p)) for p in paths]
receipt_path=OUTPUT_DIR/f'{RUN_KEY}_DRIVE_RECEIPT.json'
receipt_path.write_text(json.dumps({'status':'ACTUAL_COMPLETED_PROVIDER_BYTE_READBACK',
    'earth_engine_task_id':task.id if task else None,'raster_drive_id':file_meta['id'],
    'raster_sha256':sha256_raster,'outputs':receipts,'validation':raster_qa},indent=2))
receipts.append(upload_or_verify_unchanged(receipt_path))
display(pd.DataFrame(receipts))
print('Provider verified files, no original V05 TIFF overwritten.')

## 14. Import full 2026 municipal pixel CSV to a **new native Google Sheet** (optional)

In [ ]:
# Native Google Sheets conversion is a SECONDARY derivative; the exact CSV is preserved.
# It creates a new file, never overwrites the original V10 decision matrix or MAP002 Sheet.
IMPORT_NATIVE_SHEET=True
if IMPORT_NATIVE_SHEET:
    sheet_name=f'{RUN_KEY}_PIXELS_NATIVE_SHEET'
    candidates=drive_api.files().list(q=f"'{OUTPUT_FOLDER_ID}' in parents and name = '{sheet_name}' and trashed = false",
                 fields='files(id,name,mimeType),nextPageToken',pageSize=100).execute()['files']
    if candidates:
        raise RuntimeError('Native Sheet already exists; inspect and reuse it, not duplicate.')
    if len(rows)*8>9_500_000:
        raise RuntimeError('Native Sheet cell count may exceed Google limits; exact CSV remains authoritative.')
    imported=drive_api.files().create(body={'name':sheet_name,'mimeType':'application/vnd.google-apps.spreadsheet','parents':[OUTPUT_FOLDER_ID]},
                 media_body=MediaFileUpload(str(ledger_csv),mimetype='text/csv',resumable=True),
                 fields='id,name,mimeType,parents').execute()
    assert imported['mimeType']=='application/vnd.google-apps.spreadsheet'
    print('Native FULL PIXEL GOOGLE SHEET:', 'https://docs.google.com/spreadsheets/d/'+imported['id']+'/edit')
else:
    print('No Google Sheets conversion requested; full CSV already preserved in Drive.')

## 15. Scientific interpretation / stop conditions

- **Succeeded only if:** authenticated GEE task `COMPLETED`, GeoTIFF downloaded from the expected stable Drive folder, SHA256 recorded, class counts and coverage independently recalculated locally, and outputs read back from Drive.
- **Thematic map area of validity:** the historical municipality outline is geographic context; only colored pixels have acceptable LST; gray indicates NoData. A new citywide **raster extent** does not imply 100% **valid thermal observations**, especially in monsoon clouds.
- **Meaning of units:** temperature **°C**, geometric area **km²**, coverage **% of municipal area**, raster spacing **30 m**. The Landsat 8/9 TIRS native thermal sensor samples at approximately **100 m**, resampled in the delivered product.
- **Do not confuse:** landscape LST versus Binay's manual 1.5 m air measurements or point ground-surface readings; an image-pixel median can span different acquisition dates at neighboring pixels.
- **Still open:** the original historical municipal polygon is not independently certified against the latest municipal GIS; original source's 44-date/186-event analysis versus 47-date/202-event record requires explicit scientific interpretation; V10 researcher/supervisor sign-off cannot be generated from automated evidence.
- **If Earth Engine says 0 scenes, fails authorization or has inadequate usable pixels:** retain that result with a QA explanation; do not create a false 100%-coverage heat map.

**Historical original:** [archive 97-cell V018](https://drive.google.com/file/d/1OIqegc2RLGNqgUuO1dyrl2WclrDJAkOe/view). **Source-controlled GIS:** [GitHub](https://github.com/FabinGurung/jp-research-lab/tree/main/researchers/binay-karki/gis/map002). **Research audit:** [live Google Doc](https://docs.google.com/document/d/11EyRADuMW9ql_NhQTuFoop7BrPKFJmbBbJaMXFCt4Z0/edit).